# einops-reduce — worked example 3: Per-row min-max normalization using reduce with keepdim placeholder

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import reduce

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Using `()` on the right side of a `reduce` pattern keeps a collapsed axis as a size-1 placeholder, enabling direct broadcasting. The pattern `'b n -> b ()'` with `'min'` produces shape `(B, 1)` rather than `(B,)`. This means you can write `(x - x_min) / (x_max - x_min)` without any explicit `unsqueeze` call.

## Worked solution

Input: `(B=3, N=10)` matrix. Goal: normalize each row so its min is 0 and max is 1.

**Step 1.** `x_min = reduce(x, 'b n -> b ()', 'min')` — shape `(3, 1)`. The `()` keeps the collapsed axis as size-1.
**Step 2.** `x_max = reduce(x, 'b n -> b ()', 'max')` — shape `(3, 1)`.
**Step 3.** `normalized = (x - x_min) / (x_max - x_min + eps)` — broadcasting works because both `x_min` and `x_max` have shape `(3, 1)` which broadcasts over `(3, 10)`.

**Why `()` instead of dropping the axis?** Dropping gives shape `(3,)` — not broadcastable against `(3, 10)` without manual `unsqueeze`. The `()` placeholder saves that step.

In [ ]:
import torch as t
from einops import reduce

t.manual_seed(3)
B, N = 4, 12
x = t.randn(B, N)

def minmax_normalize(x, eps=1e-6):
    x_min = reduce(x, 'b n -> b ()', 'min')  # (B, 1)
    x_max = reduce(x, 'b n -> b ()', 'max')  # (B, 1)
    return (x - x_min) / (x_max - x_min + eps)

norm = minmax_normalize(x)
print('Input shape:', x.shape)
print('Normalized shape:', norm.shape)  # (4, 12)

# Every row should now have min ≈ 0 and max ≈ 1
row_mins = norm.min(dim=1).values
row_maxs = norm.max(dim=1).values
print('Row mins (should be ~0):', row_mins)
print('Row maxs (should be ~1):', row_maxs)
assert (row_mins < 1e-5).all()
assert ((row_maxs - 1.0).abs() < 1e-5).all()
print('All rows normalized correctly:', True)